# Join всех таблиц на `train.csv` (Favorita)

Этот ноутбук собирает единую таблицу для анализа признаков временного ряда:
- база: `train.csv`
- джойны: `stores.csv`, `items.csv`, `oil.csv`, `transactions.csv`, `holidays_events.csv`
- `submission.csv` не используется.

Код учитывает, что данные могут лежать в Google Drive (например, в Colab), но также работает и локально.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!cat '/content/drive/MyDrive/favorita-grocery-sales-forecasting (Unzipped Files)/train.csv' | wc -l

125497041


In [5]:
from io import BytesIO
from pathlib import Path
from typing import Dict

import pandas as pd


data_dir = Path('/content/drive/MyDrive/favorita-grocery-sales-forecasting (Unzipped Files)')

batch_size = 100_000

train = pd.read_csv(
    data_dir / 'train.csv',
    parse_dates=['date'],
    nrows=batch_size,
    dtype={'store_nbr': 'int16', 'item_nbr': 'int32', 'onpromotion': 'boolean'}
)

stores = pd.read_csv(
    data_dir / 'stores.csv',
    dtype={'store_nbr': 'int16'}
)

items = pd.read_csv(
    data_dir / 'items.csv',
    dtype={'item_nbr': 'int32'}
)

oil = pd.read_csv(
    data_dir / 'oil.csv',
    parse_dates=['date']
)

transactions = pd.read_csv(
    data_dir / 'transactions.csv',
    parse_dates=['date'],
    dtype={'store_nbr': 'int16'}
)

holidays = pd.read_csv(
    data_dir / 'holidays_events.csv',
    parse_dates=['date']
)

print('Shapes:')
print('train       :', train.shape)
print('stores      :', stores.shape)
print('items       :', items.shape)
print('oil         :', oil.shape)
print('transactions:', transactions.shape)
print('holidays    :', holidays.shape)

Shapes:
train       : (100000, 6)
stores      : (54, 5)
items       : (4100, 4)
oil         : (1218, 2)
transactions: (83488, 3)
holidays    : (350, 6)


In [8]:
train.columns, stores.columns, items.columns, oil.columns, transactions.columns, holidays.columns

(Index(['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion'], dtype='object'),
 Index(['store_nbr', 'city', 'state', 'type', 'cluster'], dtype='object'),
 Index(['item_nbr', 'family', 'class', 'perishable'], dtype='object'),
 Index(['date', 'dcoilwtico'], dtype='object'),
 Index(['date', 'store_nbr', 'transactions'], dtype='object'),
 Index(['date', 'type', 'locale', 'locale_name', 'description', 'transferred'], dtype='object'))

In [9]:
train.head(), stores.head(), items.head(), oil.head(), transactions.head(), holidays.head()

(   id       date  store_nbr  item_nbr  unit_sales  onpromotion
 0   0 2013-01-01         25    103665         7.0         <NA>
 1   1 2013-01-01         25    105574         1.0         <NA>
 2   2 2013-01-01         25    105575         2.0         <NA>
 3   3 2013-01-01         25    108079         1.0         <NA>
 4   4 2013-01-01         25    108701         1.0         <NA>,
    store_nbr           city                           state type  cluster
 0          1          Quito                       Pichincha    D       13
 1          2          Quito                       Pichincha    D       13
 2          3          Quito                       Pichincha    D        8
 3          4          Quito                       Pichincha    D        9
 4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4,
    item_nbr        family  class  perishable
 0     96995     GROCERY I   1093           0
 1     99197     GROCERY I   1067           0
 2    103501      CLEANING

In [7]:
# ====== 4) Подготовка holidays к джойну без раздувания train ======
# В holidays на один date бывает несколько записей; напрямую джойнить по date нельзя,
# иначе train раздуется по строкам. Ниже делаем агрегаты.
h = holidays.copy()
h['is_transferred'] = h['transferred'].fillna(False).astype(bool)
h['is_holiday_event'] = h['type'].isin(['Holiday', 'Event']).astype('int8')

# Национальные праздники/события (действуют на всю страну)
h_nat = (
    h[h['locale'].eq('National')]
    .groupby('date', as_index=False)
    .agg(
        nat_holiday_events=('is_holiday_event', 'sum'),
        nat_total_records=('type', 'size'),
        nat_any_transferred=('is_transferred', 'max')
    )
)

# Региональные: ключ (date, state)
h_reg = (
    h[h['locale'].eq('Regional')]
    .rename(columns={'locale_name': 'state'})
    .groupby(['date', 'state'], as_index=False)
    .agg(
        reg_holiday_events=('is_holiday_event', 'sum'),
        reg_total_records=('type', 'size'),
        reg_any_transferred=('is_transferred', 'max')
    )
)

# Локальные: ключ (date, city)
h_loc = (
    h[h['locale'].eq('Local')]
    .rename(columns={'locale_name': 'city'})
    .groupby(['date', 'city'], as_index=False)
    .agg(
        loc_holiday_events=('is_holiday_event', 'sum'),
        loc_total_records=('type', 'size'),
        loc_any_transferred=('is_transferred', 'max')
    )
)


# ====== 5) Джойны на train ======
# Базовая таблица
full_df = train.copy()

# stores: many-to-one по store_nbr
full_df = full_df.merge(stores, on='store_nbr', how='left', validate='many_to_one')

# items: many-to-one по item_nbr
full_df = full_df.merge(items, on='item_nbr', how='left', validate='many_to_one')

# oil: many-to-one по date
full_df = full_df.merge(oil, on='date', how='left', validate='many_to_one')

# transactions: many-to-one по (date, store_nbr)
full_df = full_df.merge(
    transactions,
    on=['date', 'store_nbr'],
    how='left',
    validate='many_to_one'
)

# holidays national (по date)
full_df = full_df.merge(h_nat, on='date', how='left', validate='many_to_one')

# holidays regional (по date + state) после stores
full_df = full_df.merge(
    h_reg,
    on=['date', 'state'],
    how='left',
    validate='many_to_one'
)

# holidays local (по date + city) после stores
full_df = full_df.merge(
    h_loc,
    on=['date', 'city'],
    how='left',
    validate='many_to_one'
)


# ====== 6) Быстрые проверки качества джойна ======
holiday_cols = [
    'nat_holiday_events', 'nat_total_records', 'nat_any_transferred',
    'reg_holiday_events', 'reg_total_records', 'reg_any_transferred',
    'loc_holiday_events', 'loc_total_records', 'loc_any_transferred',
]

for c in holiday_cols:
    if c in full_df.columns:
        if 'transferred' in c:
            full_df[c] = full_df[c].fillna(False).astype(bool)
        else:
            full_df[c] = full_df[c].fillna(0)

# Желательно оставить сортировку для временного анализа
full_df = full_df.sort_values(['date', 'store_nbr', 'item_nbr']).reset_index(drop=True)

print('Result shape:', full_df.shape)
print('Unique train keys:', train[['date', 'store_nbr', 'item_nbr']].drop_duplicates().shape[0])
print('Unique full_df keys:', full_df[['date', 'store_nbr', 'item_nbr']].drop_duplicates().shape[0])

# Пропуски после джойнов (полезно перед EDA)
missing_share = full_df.isna().mean().sort_values(ascending=False)
print('\nTop-20 missing share:')
print((missing_share.head(20) * 100).round(2).astype(str) + '%')

full_df.head()

Result shape: (100000, 24)
Unique train keys: 100000
Unique full_df keys: 100000

Top-20 missing share:
onpromotion            100.0%
dcoilwtico              0.58%
store_nbr                0.0%
id                       0.0%
item_nbr                 0.0%
unit_sales               0.0%
city                     0.0%
date                     0.0%
state                    0.0%
type                     0.0%
family                   0.0%
cluster                  0.0%
class                    0.0%
perishable               0.0%
transactions             0.0%
nat_holiday_events       0.0%
nat_total_records        0.0%
nat_any_transferred      0.0%
reg_holiday_events       0.0%
reg_total_records        0.0%
dtype: object


/tmp/ipykernel_5451/2067680146.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  full_df[c] = full_df[c].fillna(False).astype(bool)
/tmp/ipykernel_5451/2067680146.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  full_df[c] = full_df[c].fillna(False).astype(bool)
/tmp/ipykernel_5451/2067680146.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasti

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,...,transactions,nat_holiday_events,nat_total_records,nat_any_transferred,reg_holiday_events,reg_total_records,reg_any_transferred,loc_holiday_events,loc_total_records,loc_any_transferred
0,0,2013-01-01,25,103665,7.0,<NA>,Salinas,Santa Elena,D,1,...,770,1.0,1.0,False,0.0,0.0,False,0.0,0.0,False
1,1,2013-01-01,25,105574,1.0,<NA>,Salinas,Santa Elena,D,1,...,770,1.0,1.0,False,0.0,0.0,False,0.0,0.0,False
2,2,2013-01-01,25,105575,2.0,<NA>,Salinas,Santa Elena,D,1,...,770,1.0,1.0,False,0.0,0.0,False,0.0,0.0,False
3,3,2013-01-01,25,108079,1.0,<NA>,Salinas,Santa Elena,D,1,...,770,1.0,1.0,False,0.0,0.0,False,0.0,0.0,False
4,4,2013-01-01,25,108701,1.0,<NA>,Salinas,Santa Elena,D,1,...,770,1.0,1.0,False,0.0,0.0,False,0.0,0.0,False
